# This method attemps to gain a higher accuracy by training a ResNet model on Amazon-Berkley Objects dataset since the ABO dataset contains object images and their respective weight

# Download Dataset

In [23]:
import os

# 1. Download and extract listings metadata (approx 83 MB)
if not os.path.exists('listings'):
    print("Downloading listings metadata...")
    !wget https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-listings.tar
    !tar -xf abo-listings.tar
else:
    print("Listings metadata already exists, skipping download...")

# 2. Download and extract downscaled images (approx 3 GB)
if not os.path.exists('images'):
    print("Downloading downscaled images...")
    !wget https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-images-small.tar
    !tar -xf abo-images-small.tar
else:
    print("Images already exist, skipping download...")

Listings metadata already exists, skipping download...
Images already exist, skipping download...


In [24]:
import torch
import torchvision.models as models
from torch import nn
import torch.nn.functional as F
from typing import Dict, Union, List

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [26]:
import pandas as pd

# Load the listings metadata
df = pd.read_json('listings/metadata/listings_0.json.gz', lines=True, compression='gzip')

# Load the images metadata
images_df = pd.read_csv('images/metadata/images.csv.gz', compression='gzip')

# Join metadata with images using image_id
# For main images:
df_with_main_image = df.merge(
    images_df,
    left_on='main_image_id',
    right_on='image_id',
    how='left'
)

# Extract weight from nested structure
df['weight_pounds'] = df['item_weight'].apply(
    lambda x: x[0]['normalized_value']['value'] if isinstance(x, list) and len(x) > 0 else None
)
print(f"Number of entries with weight: {df['weight_pounds'].notna().sum()} out of {len(df)}")
print(df.head())

# Create full image path
df_with_main_image['image_full_path'] = 'images/small/' + df_with_main_image['path'].astype(str)

# Create training dataframe
training_df = df_with_main_image[['main_image_id', 'item_id', 'weight_pounds', 'image_full_path']].copy()

# Remove entries missing weight
training_df = training_df.dropna(subset=['weight_pounds'])

print(f"Training dataframe shape after removing missing weights: {training_df.shape}")
print(training_df.head())

Number of entries with weight: 6623 out of 9232
                                               brand  \
0      [{'language_tag': 'nl_NL', 'value': 'find.'}]   
1  [{'language_tag': 'es_MX', 'value': 'AmazonBas...   
2  [{'language_tag': 'en_AE', 'value': 'AmazonBas...   
3  [{'language_tag': 'en_GB', 'value': 'Stone & B...   
4    [{'language_tag': 'en_AU', 'value': 'The Fix'}]   

                                        bullet_point  \
0  [{'language_tag': 'nl_NL', 'value': 'Schoen in...   
1  [{'language_tag': 'es_MX', 'value': 'White Pow...   
2  [{'language_tag': 'en_AE', 'value': '3D printe...   
3                                                NaN   
4  [{'language_tag': 'en_AU', 'value': 'Embroider...   

                                               color     item_id  \
0  [{'language_tag': 'nl_NL', 'value': 'Veelkleur...  B06X9STHNG   
1  [{'language_tag': 'es_MX', 'value': 'White Pow...  B07P8ML82R   
2  [{'language_tag': 'en_AE', 'value': 'Transluce...  B07H9GMYXS   
3  [{'

KeyError: "['weight_pounds'] not in index"